# LeetCode 752 — Open the Lock bằng Python

Link đề: https://leetcode.com/problems/open-the-lock/ · Độ khó: **Medium** · Chủ đề: Breadth-First Search (BFS), Hash Table, String, Bidirectional Search (tìm kiếm hai chiều)

Skeleton LeetCode:

```python
class Solution:
    def openLock(self, deadends: List[str], target: str) -> int:
        ...
```

Ghi chú: bài là mẫu kinh điển của **BFS trên đồ thị ngầm** — ta không được cho sẵn đồ thị, mà mỗi "đỉnh" là một trạng thái của ổ khoá (`"0000"`..`"9999"`, tổng cộng `10^4` đỉnh) và hàng xóm của nó là các trạng thái sau một lần xoay. Notebook đi từ cách lặp cập nhật toàn bộ bảng khoảng cách, tới BFS chuẩn, BFS hai chiều và A* (tìm kiếm có heuristic).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ổ khoá 4 bánh xe, mỗi bánh có các chữ số `0..9` xoay vòng, mỗi lần xoay một bánh một nấc; bắt đầu từ `"0000"`, gặp "ngõ cụt" (`deadends`) thì khoá kẹt; tìm số lần xoay **ít nhất** tới `target`, hoặc `-1`.
2. Nắm các công cụ Python cần dùng: chuỗi (cắt/ghép), phép `% 10` để xoay vòng, `set` để tra cứu `O(1)`, `collections.deque` (hàng đợi hai đầu), `heapq` (hàng đợi ưu tiên) và các hàm dựng hàng xóm.
3. Đi từ **lặp cập nhật khoảng cách trên mọi trạng thái** → **BFS chuẩn** → **BFS hai chiều** → **A\*** và hiểu vì sao BFS cho đường đi ngắn nhất khi mọi bước có chi phí bằng nhau.
4. Hiểu mấu chốt: **mô hình hoá thành đồ thị** (đỉnh = trạng thái, cạnh = một lần xoay, `deadends` = các đỉnh bị xoá) và **đánh dấu `visited` ngay khi đưa vào hàng đợi**.
5. Tránh các lỗi kinh điển: quên `"0000"` có thể nằm trong `deadends`, đánh dấu `visited` muộn (đưa trùng vào hàng đợi), chỉ xoay một chiều (quên vòng `9 → 0` và `0 → 9`), dùng DFS thay BFS, dùng `str.replace` để đổi chữ số.

## 2. Đề bài & ví dụ

Có một ổ khoá với **4 bánh xe tròn**. Mỗi bánh có 10 vị trí: `'0','1',...,'9'`. Các bánh xoay tự do và **quay vòng**: `'9'` có thể xoay thành `'0'` và `'0'` có thể xoay thành `'9'`. Mỗi bước đi là xoay **một bánh một nấc**.

Ổ khoá ban đầu hiển thị `"0000"` (chuỗi biểu diễn trạng thái 4 bánh).

Cho danh sách `deadends` (các "ngõ cụt"): nếu ổ khoá hiển thị một mã nằm trong danh sách này thì các bánh dừng xoay và bạn **không thể mở khoá nữa**.

Cho `target` là giá trị cần đạt để mở khoá, trả về **số lần xoay ít nhất** để mở khoá, hoặc `-1` nếu không thể.

**Ví dụ 1**

- Input: `deadends = ["0201","0101","0102","1212","2002"], target = "0202"`
- Output: `6`
- Một dãy hợp lệ: `"0000" → "1000" → "1100" → "1200" → "1201" → "1202" → "0202"`. Dãy `"0000" → "0001" → "0002" → "0102" → "0202"` **không hợp lệ**, vì `"0102"` là ngõ cụt nên khoá kẹt ở đó.

**Ví dụ 2**

- Input: `deadends = ["8888"], target = "0009"`
- Output: `1` (xoay ngược bánh cuối: `"0000" → "0009"`).

**Ví dụ 3**

- Input: `deadends = ["8887","8889","8878","8898","8788","8988","7888","9888"], target = "8888"`
- Output: `-1` (tám trạng thái quanh `"8888"` đều là ngõ cụt nên không thể tới `"8888"`).

**Ràng buộc:**

- `1 <= deadends.length <= 500`
- `deadends[i].length == 4`, `target.length == 4`
- `target` **không** nằm trong `deadends`
- `target` và `deadends[i]` chỉ gồm chữ số.

Lưu ý quan trọng (không có trong ràng buộc): `"0000"` **có thể** nằm trong `deadends`; khi đó ta kẹt ngay từ đầu và đáp án là `-1`.

In [1]:
from typing import List
from collections import deque
import heapq
import random

examples = [
    (["0201", "0101", "0102", "1212", "2002"], "0202", 6),
    (["8888"], "0009", 1),
    (["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888", -1),
]
for dead, target, expected in examples:
    print(f"deadends = {dead} | target = {target} | kỳ vọng {expected}")

deadends = ['0201', '0101', '0102', '1212', '2002'] | target = 0202 | kỳ vọng 6
deadends = ['8888'] | target = 0009 | kỳ vọng 1
deadends = ['8887', '8889', '8878', '8898', '8788', '8988', '7888', '9888'] | target = 8888 | kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Chuỗi: cắt, ghép và đổi một ký tự

Chuỗi Python **không sửa được tại chỗ**; muốn đổi ký tự thứ `i` ta ghép lại: `s[:i] + ký_tự_mới + s[i+1:]`. Với `s = "0123"` và `i = 2`: `s[:2] = "01"`, `s[3:] = "3"`, ghép với `"9"` được `"0193"`.

Ngoài ra `int("7")` đổi ký tự chữ số thành số, `str(7)` đổi ngược lại.

In [2]:
s = "0123"
i = 2
print(s[:i], "|", s[i], "|", s[i + 1:])        # 01 | 2 | 3
print(s[:i] + "9" + s[i + 1:])                  # 0193
print(int(s[i]) + 1, str(int(s[i]) + 1))

01 | 2 | 3
0193
3 3


### 3.2. Xoay vòng bằng `% 10`

Bánh xe vòng tròn 10 nấc: sau `9` là `0`, trước `0` là `9`. Với chữ số `d`, xoay lên là `(d + 1) % 10`, xoay xuống là `(d - 1) % 10`. Python cho kết quả **không âm** khi lấy dư số âm: `(-1) % 10 == 9` (ở nhiều ngôn ngữ khác thì không, nên cẩn thận khi chuyển sang Java/C++: dùng `(d + 9) % 10`).

In [3]:
print([(d + 1) % 10 for d in range(10)])       # 1..9, 0
print([(d - 1) % 10 for d in range(10)])       # 9, 0..8
print((-1) % 10, (0 - 1) % 10, (9 + 1) % 10)   # 9 9 0

[1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
[9, 0, 1, 2, 3, 4, 5, 6, 7, 8]
9 9 0


### 3.3. Hàm sinh hàng xóm

Từ một trạng thái `s` (chuỗi 4 chữ số), có `4 bánh × 2 chiều = 8` trạng thái kề. Ta viết hàm `neighbors(s)` trả về chúng. Đây chính là "danh sách kề" của đồ thị ngầm.

In [4]:
def neighbors(s):
    result = []
    for i in range(4):
        d = int(s[i])
        for delta in (1, -1):
            nd = (d + delta) % 10
            result.append(s[:i] + str(nd) + s[i + 1:])
    return result

print(neighbors("0000"))
print(neighbors("9090"))
print(len(neighbors("5555")))     # 8

['1000', '9000', '0100', '0900', '0010', '0090', '0001', '0009']
['0090', '8090', '9190', '9990', '9000', '9080', '9091', '9099']
8


### 3.4. `set` để tra cứu nhanh, và `deque` làm hàng đợi

- `x in some_set` là `O(1)` trung bình, còn `x in some_list` là `O(n)`. Ta đổi `deadends` (list) thành `set` để kiểm tra ngõ cụt nhanh.
- `collections.deque` cho `append` (thêm cuối) và `popleft` (lấy đầu) đều `O(1)`. Còn `list.pop(0)` là `O(n)` nên không dùng làm hàng đợi.
- **Hàng đợi** là cấu trúc "vào trước ra trước" (FIFO): đúng thứ BFS cần.

In [5]:
dead = ["0201", "0101"]
dead_set = set(dead)
print("0201" in dead_set, "9999" in dead_set)

q = deque()
q.append("a"); q.append("b"); q.append("c")
print(q.popleft(), q.popleft(), list(q))       # a b ['c']

True False
a b ['c']


### 3.5. Duyệt theo tầng (BFS) — nhắc lại ý tưởng

**BFS (Breadth-First Search — duyệt theo chiều rộng)**: bắt đầu từ đỉnh xuất phát (tầng 0), thăm hết hàng xóm của nó (tầng 1), rồi hàng xóm chưa thăm của tầng 1 (tầng 2), ... Vì thăm theo tầng tăng dần nên **lần đầu** tới một đỉnh là bằng **đường đi ngắn nhất** (theo số cạnh).

Ví dụ nhỏ: đồ thị `a-b, a-c, b-d, c-d, d-e`. Từ `a`: tầng 0 `{a}`, tầng 1 `{b, c}`, tầng 2 `{d}`, tầng 3 `{e}`. Đường ngắn nhất `a → e` có 3 cạnh.

In [6]:
graph = {"a": ["b", "c"], "b": ["a", "d"], "c": ["a", "d"], "d": ["b", "c", "e"], "e": ["d"]}
dist = {"a": 0}
queue = deque(["a"])
while queue:
    u = queue.popleft()
    for v in graph[u]:
        if v not in dist:             # chưa thăm
            dist[v] = dist[u] + 1
            queue.append(v)
print(dist)                            # {'a': 0, 'b': 1, 'c': 1, 'd': 2, 'e': 3}

{'a': 0, 'b': 1, 'c': 1, 'd': 2, 'e': 3}


## 4. Ý tưởng nền tảng / trực giác

**Đỉnh và cạnh.** Mỗi mã `abcd` là một đỉnh; hai mã là hàng xóm nếu khác nhau đúng một bánh và bánh đó chênh nhau **một nấc** (xoay vòng). Mỗi đỉnh có 8 hàng xóm. Có `10^4 = 10 000` đỉnh. Các ngõ cụt là các đỉnh ta **không được bước vào** (thực ra vào được nhưng kẹt — vì đích không phải ngõ cụt nên bước vào ngõ cụt chẳng bao giờ có ích, coi như xoá đỉnh đó).

**Khi không có ngõ cụt:** mỗi bánh độc lập. Số nấc tối thiểu để đưa chữ số `a` về `b` trên vòng 10 là `min(|a - b|, 10 - |a - b|)`. Tổng qua 4 bánh là đáp án (một **cận dưới** khi có ngõ cụt!). Ví dụ `"0202"`: `0→0: 0`, `0→2: 2`, `0→0: 0`, `0→2: 2` → `4`. Với ngõ cụt ở Ví dụ 1, đáp án là `6`, lớn hơn cận dưới `4`: ngõ cụt buộc ta đi vòng.

Tính tay Ví dụ 2: `"0000" → "0009"`: bánh cuối `0 → 9` chỉ `min(9, 1) = 1` nấc (xoay ngược). `"8888"` là ngõ cụt nhưng không cản đường. Đáp án `1`.

In [7]:
def circular_distance(a, b):
    d = abs(int(a) - int(b))
    return min(d, 10 - d)

def lower_bound(s, t):
    # số nấc tối thiểu nếu KHÔNG có ngõ cụt
    return sum(circular_distance(x, y) for x, y in zip(s, t))

print(lower_bound("0000", "0202"))   # 4
print(lower_bound("0000", "0009"))   # 1
print(lower_bound("0000", "5555"))   # 20 (xa nhất)
print(lower_bound("0000", "8888"))   # 8

4
1
20
8


## 5. Cách 1 — Brute Force: lặp cập nhật khoảng cách trên mọi trạng thái

### Ý tưởng

Không cần hàng đợi: lập bảng `dist[0..9999]` (số lần xoay ít nhất từ `"0000"`), ban đầu `dist["0000"] = 0`, còn lại `∞`. Lặp: với mỗi trạng thái `u` đã có `dist[u]` hữu hạn và không phải ngõ cụt, cập nhật mỗi hàng xóm `v` (không phải ngõ cụt): `dist[v] = min(dist[v], dist[u] + 1)`. **Lặp đến khi không còn gì thay đổi** (giống thuật toán Bellman-Ford). Cuối cùng `dist[target]` là đáp án (hoặc `-1` nếu vẫn `∞`).

Đơn giản và chắc chắn đúng (nó chỉ dùng định nghĩa "đường ngắn nhất"), nhưng chậm: mỗi lượt quét toàn bộ `10^4` trạng thái, và cần tới (độ dài đường dài nhất) lượt.

### Code

Để chạy nhanh hơn, ta dựng **bảng hàng xóm một lần** dưới dạng số nguyên `0..9999`.

In [8]:
# Bảng hàng xóm cho mọi trạng thái, dựng một lần: NEI[u] = danh sách 8 hàng xóm (dạng số nguyên)
def build_neighbor_table():
    table = []
    for u in range(10000):
        s = f"{u:04d}"
        table.append([int(v) for v in neighbors(s)])
    return table

NEI = build_neighbor_table()
print(NEI[0])            # hàng xóm của 0000: 1000, 9000, 0100, 0900, 0010, 0090, 0001, 0009 (dạng số)
print(len(NEI), "trạng thái;", len(NEI[1234]), "hàng xóm mỗi trạng thái")

[1000, 9000, 100, 900, 10, 90, 1, 9]
10000 trạng thái; 8 hàng xóm mỗi trạng thái


In [9]:
INF = float("inf")

def open_lock_relax(deadends, target):
    dead = {int(x) for x in deadends}
    t = int(target)
    if 0 in dead:
        return -1
    dist = [INF] * 10000
    dist[0] = 0
    changed = True
    while changed:                              # lặp đến khi hết cải thiện
        changed = False
        for u in range(10000):
            du = dist[u]
            if du == INF:
                continue
            for v in NEI[u]:
                if v not in dead and dist[v] > du + 1:
                    dist[v] = du + 1
                    changed = True
    return -1 if dist[t] == INF else dist[t]

for dead, target, expected in examples:
    got = open_lock_relax(dead, target)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
1 1 OK


-1 -1 OK


### Độ phức tạp và nút thắt

- **Time:** `O(L · N · 8)` với `N = 10^4` là số trạng thái, `L` là số lượt lặp (cỡ độ dài đường ngắn nhất lớn nhất, vài chục). Khoảng vài triệu phép: chạy được (vài giây ở Python) nhưng lãng phí.
- **Space:** `O(N)`.

**Nút thắt:** mỗi lượt ta quét **mọi** trạng thái kể cả những trạng thái chưa tới được hoặc đã xử lý xong. BFS khắc phục: chỉ xử lý mỗi trạng thái **đúng một lần**, theo thứ tự khoảng cách tăng dần. Cách 1 vẫn rất hữu ích làm **oracle độc lập** (nó không dùng ý tưởng BFS).

## 6. Ý tưởng thuật toán / bước đột phá

Mọi bước đi có **chi phí bằng nhau** (1 lần xoay). Với đồ thị không trọng số, **BFS** cho đường ngắn nhất:

1. Hàng đợi chứa các trạng thái cùng khoảng cách `d` (rồi `d + 1`, ...): BFS thăm **tầng 0, tầng 1, tầng 2,...** theo đúng thứ tự.
2. Mỗi trạng thái chỉ được **đưa vào hàng đợi một lần** (lần đầu gặp = lần ngắn nhất). Ta ghi nhớ bằng tập `visited`.
3. Ngõ cụt xử lý bằng cách **cho vào `visited` ngay từ đầu**: như vậy BFS không bao giờ đi vào chúng. Trường hợp `"0000"` là ngõ cụt: trả `-1` ngay.

Có thể tăng tốc nhờ **tìm kiếm hai chiều** (xuất phát từ cả `"0000"` và `target`, gặp nhau ở giữa) hay **A\*** (dùng cận dưới `lower_bound` ở mục 4 làm heuristic dẫn đường).

Cell dưới đếm số trạng thái ở từng tầng khi không có ngõ cụt: `1, 8, 32, ...` — để thấy BFS phình rất nhanh (và vì sao hai chiều có lợi).

In [10]:
def level_sizes(deadends=()):
    dead = set(deadends)
    seen = {"0000"} | dead
    level = ["0000"]
    sizes = [1]
    while True:
        nxt = []
        for s in level:
            for t in neighbors(s):
                if t not in seen:
                    seen.add(t)
                    nxt.append(t)
        if not nxt:
            break
        sizes.append(len(nxt))
        level = nxt
    return sizes

sz = level_sizes()
print(sz)
print("tổng =", sum(sz), "(đủ 10^4 trạng thái); tầng sâu nhất =", len(sz) - 1, "(= 5+5+5+5 = 20 là xa nhất)")

[1, 8, 32, 88, 192, 356, 576, 824, 1056, 1224, 1286, 1224, 1056, 824, 576, 356, 192, 88, 32, 8, 1]
tổng = 10000 (đủ 10^4 trạng thái); tầng sâu nhất = 20 (= 5+5+5+5 = 20 là xa nhất)


## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS chuẩn bằng `deque` (lời giải nộp)

**Giải thích từng đoạn.**

1. `dead = set(deadends)`; nếu `"0000"` thuộc `dead` → `-1`. Nếu `target == "0000"` → `0`.
2. `visited = dead ∪ {"0000"}`; hàng đợi chứa `("0000", 0)` (trạng thái, số lần xoay).
3. Lặp: lấy `(s, d)` ở đầu hàng đợi; với mỗi hàng xóm `t` chưa thuộc `visited`: nếu `t == target` trả `d + 1`; ngược lại đánh dấu `visited` **ngay** rồi đưa `(t, d + 1)` vào cuối hàng đợi.
4. Hàng đợi cạn mà chưa tới `target` → `-1`.

**Code demo:**

In [11]:
def open_lock_bfs(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1
    if target == "0000":
        return 0
    visited = dead | {"0000"}
    queue = deque([("0000", 0)])
    while queue:
        s, d = queue.popleft()
        for t in neighbors(s):
            if t in visited:
                continue
            if t == target:
                return d + 1
            visited.add(t)                 # đánh dấu NGAY khi đưa vào hàng đợi
            queue.append((t, d + 1))
    return -1

for dead, target, expected in examples:
    got = open_lock_bfs(dead, target)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
1 1 OK
-1 -1 OK


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionBFS:
    def openLock(self, deadends: List[str], target: str) -> int:
        dead = set(deadends)
        if "0000" in dead:
            return -1
        if target == "0000":
            return 0

        def nxt(s: str) -> List[str]:
            res = []
            for i in range(4):
                d = int(s[i])
                for delta in (1, -1):
                    res.append(s[:i] + str((d + delta) % 10) + s[i + 1:])
            return res

        visited = dead | {"0000"}
        queue = deque([("0000", 0)])
        while queue:
            s, d = queue.popleft()
            for t in nxt(s):
                if t in visited:
                    continue
                if t == target:
                    return d + 1
                visited.add(t)
                queue.append((t, d + 1))
        return -1

print(SolutionBFS().openLock(["8888"], "0009"))   # 1

1


**Độ phức tạp (Cách 2):** Time `O(N · 8 · 4)` với `N = 10^4` (mỗi trạng thái xử lý một lần, sinh 8 hàng xóm, mỗi hàng xóm tốn `O(4)` để dựng chuỗi) ≈ `3.2 · 10^5` phép cơ bản. Space `O(N)` cho `visited` và hàng đợi.

### 7.2. Cách 3 — BFS hai chiều (bidirectional BFS)

**Giải thích từng đoạn.**

1. Giữ **hai tập** `begin = {"0000"}` và `end = {target}`; `visited` chứa cả ngõ cụt và các trạng thái đã thăm từ mỗi phía.
2. Mỗi vòng lặp **mở rộng tập nhỏ hơn** (để giữ hai phía cân bằng): với mỗi `s` trong tập đó, sinh hàng xóm `t`: nếu `t` thuộc tập **bên kia** thì hai phía gặp nhau → trả về `steps + 1`; nếu chưa thăm thì thêm vào tập mới.
3. Thay tập đang mở rộng bằng tập mới, tăng `steps`. Nếu một tập rỗng thì không thể gặp → `-1`.

**Vì sao nhanh hơn:** BFS một chiều phải duyệt hết các trạng thái đến độ sâu `D`; hai chiều chỉ cần mỗi phía tới độ sâu `~D/2`. Do kích thước tầng tăng gần như hàm mũ theo độ sâu (mục 6), tổng số trạng thái thăm giảm đáng kể. Khi `target` bị bao quanh bởi ngõ cụt (Ví dụ 3), phía `target` hết trạng thái ngay lập tức nên dừng ngay.

**Code demo:**

In [13]:
def open_lock_bidirectional(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1
    if target == "0000":
        return 0
    begin, end = {"0000"}, {target}
    visited = set(dead)
    steps = 0
    while begin and end:
        if len(begin) > len(end):              # luôn mở rộng phía nhỏ hơn
            begin, end = end, begin
        nxt = set()
        for s in begin:
            for t in neighbors(s):
                if t in end:                   # hai phía gặp nhau
                    return steps + 1
                if t not in visited and t not in begin:
                    nxt.add(t)
        visited |= begin                       # tầng vừa mở rộng đã xong
        begin = nxt
        steps += 1
    return -1

for dead, target, expected in examples:
    got = open_lock_bidirectional(dead, target)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
1 1 OK
-1 -1 OK


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionBidirectional:
    def openLock(self, deadends: List[str], target: str) -> int:
        dead = set(deadends)
        if "0000" in dead:
            return -1
        if target == "0000":
            return 0

        def nxt(s: str) -> List[str]:
            res = []
            for i in range(4):
                d = int(s[i])
                for delta in (1, -1):
                    res.append(s[:i] + str((d + delta) % 10) + s[i + 1:])
            return res

        begin, end = {"0000"}, {target}
        visited = set(dead)
        steps = 0
        while begin and end:
            if len(begin) > len(end):
                begin, end = end, begin
            new_level = set()
            for s in begin:
                for t in nxt(s):
                    if t in end:
                        return steps + 1
                    if t not in visited and t not in begin:
                        new_level.add(t)
            visited |= begin
            begin = new_level
            steps += 1
        return -1

print(SolutionBidirectional().openLock(["8888"], "0009"))   # 1

1


**Độ phức tạp (Cách 3):** trong trường hợp xấu nhất vẫn `O(N · 8 · 4)` (khi hai phía không gặp nhau và phải duyệt hết), nhưng thực tế thường thăm ít trạng thái hơn nhiều. Space `O(N)`.

Một chi tiết dễ sai: sau khi hoán đổi `begin ↔ end`, `steps` vẫn tính **tổng số lần xoay** vì mỗi lần mở rộng một phía thêm đúng 1 cạnh vào đường đi cuối cùng.

### 7.3. Cách 4 — A\* (tìm kiếm có heuristic)

**Giải thích từng đoạn.** A\* thăm trạng thái theo thứ tự `f = g + h`, trong đó:

- `g` = số lần xoay đã dùng để tới trạng thái đó,
- `h` = **ước lượng số lần xoay còn lại** = `lower_bound(s, target)` (tổng khoảng cách vòng của 4 bánh) — luôn **không vượt quá** chi phí thật (vì ngõ cụt chỉ làm đường dài thêm), và mỗi lần xoay giảm `h` tối đa 1 nên heuristic **nhất quán (consistent)**.

Dùng `heapq` (hàng đợi ưu tiên) lấy trạng thái có `f` nhỏ nhất. Khi lấy ra `target` thì `g` chính là đáp án. Nhờ heuristic, A\* đi thẳng về phía đích và thường thăm ít trạng thái hơn BFS; nhưng với đồ thị nhỏ như bài này lợi thế không lớn, và hằng số của heap lớn hơn.

**Code demo:**

In [15]:
def open_lock_astar(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1
    if target == "0000":
        return 0

    def h(s):
        return sum(min(abs(int(a) - int(b)), 10 - abs(int(a) - int(b))) for a, b in zip(s, target))

    best = {"0000": 0}
    heap = [(h("0000"), 0, "0000")]            # (f, g, state)
    while heap:
        f, g, s = heapq.heappop(heap)
        if s == target:
            return g
        if g > best.get(s, INF):               # bản ghi cũ đã bị thay thế
            continue
        for t in neighbors(s):
            if t in dead:
                continue
            ng = g + 1
            if ng < best.get(t, INF):
                best[t] = ng
                heapq.heappush(heap, (ng + h(t), ng, t))
    return -1

for dead, target, expected in examples:
    got = open_lock_astar(dead, target)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
1 1 OK
-1 -1 OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionAStar:
    def openLock(self, deadends: List[str], target: str) -> int:
        dead = set(deadends)
        if "0000" in dead:
            return -1
        if target == "0000":
            return 0

        def h(s: str) -> int:
            total = 0
            for a, b in zip(s, target):
                d = abs(int(a) - int(b))
                total += min(d, 10 - d)
            return total

        best = {"0000": 0}
        heap = [(h("0000"), 0, "0000")]
        while heap:
            f, g, s = heapq.heappop(heap)
            if s == target:
                return g
            if g > best.get(s, 10 ** 9):
                continue
            for i in range(4):
                d = int(s[i])
                for delta in (1, -1):
                    t = s[:i] + str((d + delta) % 10) + s[i + 1:]
                    if t in dead:
                        continue
                    ng = g + 1
                    if ng < best.get(t, 10 ** 9):
                        best[t] = ng
                        heapq.heappush(heap, (ng + h(t), ng, t))
        return -1

print(SolutionAStar().openLock(["0201", "0101", "0102", "1212", "2002"], "0202"))   # 6

6


**Độ phức tạp (Cách 4):** Time `O(N · 8 · log N)` trong trường hợp xấu (thêm `log` của heap), thực tế thường thăm ít hơn BFS; Space `O(N)`.

## 8. TRACE từng bước

Trace Cách 2 (BFS) cho Ví dụ 1: `deadends = ["0201","0101","0102","1212","2002"]`, `target = "0202"`. Hàm `trace_bfs` in từng **tầng**: số trạng thái, vài trạng thái đầu, và những trạng thái bị loại vì là ngõ cụt khi xuất hiện.

Bảng trạng thái (rút gọn):

| tầng d | số trạng thái trong hàng đợi | ghi chú |
|---|---|---|
| 0 | 1 (`0000`) | xuất phát |
| 1 | 8 | `1000, 9000, 0100, 0900, 0010, 0090, 0001, 0009` |
| 2 | ~ 32 trừ ngõ cụt | ví dụ `0101` bị loại (ngõ cụt) |
| ... | | |
| 6 | chứa `0202` | trả về `6` |

In [17]:
def trace_bfs(deadends, target, show=6):
    dead = set(deadends)
    print("deadends:", sorted(dead), "| target:", target)
    if "0000" in dead:
        print("0000 là ngõ cụt -> -1")
        return -1
    visited = dead | {"0000"}
    level = ["0000"]
    d = 0
    print(f"tầng 0: 1 trạng thái ['0000']")
    while level:
        nxt = []
        blocked = []
        for s in level:
            for t in neighbors(s):
                if t in dead:
                    blocked.append(t)
                    continue
                if t in visited:
                    continue
                if t == target:
                    print(f"tầng {d + 1}: gặp target {target} từ {s} -> trả về {d + 1}")
                    return d + 1
                visited.add(t)
                nxt.append(t)
        d += 1
        print(f"tầng {d}: {len(nxt)} trạng thái mới, vd {nxt[:show]} | gặp ngõ cụt {len(set(blocked))} lần: {sorted(set(blocked))[:4]}")
        level = nxt
    print("hàng đợi cạn -> -1")
    return -1

trace_bfs(["0201", "0101", "0102", "1212", "2002"], "0202")

deadends: ['0101', '0102', '0201', '1212', '2002'] | target: 0202
tầng 0: 1 trạng thái ['0000']
tầng 1: 8 trạng thái mới, vd ['1000', '9000', '0100', '0900', '0010', '0090'] | gặp ngõ cụt 0 lần: []
tầng 2: 31 trạng thái mới, vd ['2000', '1100', '1900', '1010', '1090', '1001'] | gặp ngõ cụt 1 lần: ['0101']
tầng 3: 86 trạng thái mới, vd ['3000', '2100', '2900', '2010', '2090', '2001'] | gặp ngõ cụt 2 lần: ['0102', '0201']
tầng 4: 190 trạng thái mới, vd ['4000', '3100', '3900', '3010', '3090', '3001'] | gặp ngõ cụt 2 lần: ['0101', '2002']
tầng 5: 356 trạng thái mới, vd ['5000', '4100', '4900', '4010', '4090', '4001'] | gặp ngõ cụt 2 lần: ['0102', '0201']
tầng 6: gặp target 0202 từ 1202 -> trả về 6


6

Trace thứ hai: Ví dụ 3 (target bị bao vây). BFS duyệt gần hết `10^4` trạng thái rồi mới kết luận `-1`; ta chỉ in số trạng thái mỗi tầng (không in từng trạng thái).

In [18]:
def trace_bfs_counts(deadends, target):
    dead = set(deadends)
    visited = dead | {"0000"}
    level = ["0000"]
    d = 0
    counts = [1]
    while level:
        nxt = []
        for s in level:
            for t in neighbors(s):
                if t in visited:
                    continue
                if t == target:
                    return d + 1, counts
                visited.add(t)
                nxt.append(t)
        d += 1
        if nxt:
            counts.append(len(nxt))
        level = nxt
    return -1, counts

ans, counts = trace_bfs_counts(examples[2][0], examples[2][1])
print("đáp án:", ans)
print("số trạng thái mỗi tầng:", counts)
print("tổng số trạng thái đã thăm:", sum(counts), "(8 ngõ cụt không được thăm, và '8888' không tới được)")

đáp án: -1
số trạng thái mỗi tầng: [1, 8, 32, 88, 192, 356, 576, 820, 1055, 1220, 1286, 1224, 1056, 824, 576, 356, 192, 88, 32, 8, 1]
tổng số trạng thái đã thăm: 9991 (8 ngõ cụt không được thăm, và '8888' không tới được)


Trace thứ ba: BFS hai chiều cho Ví dụ 1, in kích thước hai tập qua từng vòng và phía nào được mở rộng.

In [19]:
def trace_bidirectional(deadends, target):
    dead = set(deadends)
    begin, end = {"0000"}, {target}
    visited = set(dead)
    steps = 0
    forward = True
    while begin and end:
        if len(begin) > len(end):
            begin, end = end, begin
            forward = not forward
        side = "từ 0000" if forward else "từ target"
        print(f"vòng {steps + 1}: mở rộng phía {side}; |begin| = {len(begin)}, |end| = {len(end)}")
        nxt = set()
        for s in begin:
            for t in neighbors(s):
                if t in end:
                    print(f"  hai phía gặp nhau tại {t} -> đáp án {steps + 1}")
                    return steps + 1
                if t not in visited and t not in begin:
                    nxt.add(t)
        visited |= begin
        begin = nxt
        steps += 1
    print("một phía cạn -> -1")
    return -1

trace_bidirectional(examples[0][0], examples[0][1])

vòng 1: mở rộng phía từ 0000; |begin| = 1, |end| = 1
vòng 2: mở rộng phía từ target; |begin| = 1, |end| = 8
vòng 3: mở rộng phía từ target; |begin| = 6, |end| = 8
vòng 4: mở rộng phía từ 0000; |begin| = 8, |end| = 28
vòng 5: mở rộng phía từ target; |begin| = 28, |end| = 31
vòng 6: mở rộng phía từ 0000; |begin| = 31, |end| = 84
  hai phía gặp nhau tại 9200 -> đáp án 6


6

## 9. Code nộp LeetCode

Lời giải chuẩn: BFS bằng `deque` (Cách 2) — ngắn gọn, dễ chứng minh, đủ nhanh (`~3 · 10^5` phép). Đúng `class Solution` và signature.

In [20]:
from typing import List
from collections import deque

class Solution:
    def openLock(self, deadends: List[str], target: str) -> int:
        dead = set(deadends)
        if "0000" in dead:                 # kẹt ngay từ đầu
            return -1
        if target == "0000":
            return 0

        def neighbors(s: str) -> List[str]:
            res = []
            for i in range(4):
                d = int(s[i])
                for delta in (1, -1):      # xoay lên / xuống, quay vòng bằng % 10
                    res.append(s[:i] + str((d + delta) % 10) + s[i + 1:])
            return res

        visited = dead | {"0000"}          # ngõ cụt coi như đã thăm: không bao giờ đi vào
        queue = deque([("0000", 0)])
        while queue:
            s, d = queue.popleft()
            for t in neighbors(s):
                if t in visited:
                    continue
                if t == target:
                    return d + 1
                visited.add(t)             # đánh dấu ngay khi đưa vào hàng đợi
                queue.append((t, d + 1))
        return -1

sol = Solution()
for dead, target, expected in examples:
    got = sol.openLock(dead, target)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
1 1 OK


-1 -1 OK


## 10. Vì sao đúng

**Mô hình.** Đồ thị `G` có đỉnh là 10^4 mã `abcd`, cạnh nối hai mã khác nhau đúng một bánh và chênh một nấc (xoay vòng). Cạnh vô hướng (xoay lên rồi xuống trả lại). Các ngõ cụt bị xoá khỏi `G` (đi vào chúng là hết đường; vì `target` không phải ngõ cụt nên một đường tối ưu không cần đi qua chúng). Đáp án là **khoảng cách ngắn nhất** từ `"0000"` tới `target` trong `G` (hoặc `-1` nếu không có đường).

**Tính đúng của BFS (quy nạp theo tầng).** Gọi `L_d` là tập đỉnh có khoảng cách đúng `d`. Mệnh đề: khi BFS xử lý xong tầng `d`, hàng đợi chứa đúng `L_{d+1}`.

- `d = 0`: `L_0 = {0000}`.
- Giả sử đúng đến `d`. Một đỉnh `v` có khoảng cách `d + 1` khi và chỉ khi có hàng xóm `u ∈ L_d` và `v` chưa thuộc `L_0..L_d`. BFS duyệt mọi `u ∈ L_d`, sinh hàng xóm; những đỉnh chưa `visited` (tức không thuộc `L_0..L_d` và không phải ngõ cụt) được thêm vào tầng `d + 1`. Vậy tầng mới đúng bằng `L_{d+1}`.

Do đó lần đầu ta gặp `target` là ở tầng nhỏ nhất, đúng đáp án; nếu không gặp thì không tới được → `-1`.

**Vì sao đánh dấu `visited` khi đưa vào hàng đợi:** để mỗi đỉnh vào hàng đợi đúng một lần. Nếu chỉ đánh dấu khi lấy ra, cùng một đỉnh có thể bị đưa vào nhiều lần từ nhiều đỉnh khác nhau (mục 13 đo số lần).

**Hai chiều.** Đồ thị vô hướng nên đường `0000 → target` đảo ngược là đường `target → 0000`. Khi một đỉnh sinh ra từ phía này thuộc tập biên của phía kia, ghép hai nửa đường ta được đường có độ dài `steps + 1`; vì hai phía đều mở rộng theo tầng, đây là đường ngắn nhất (mở rộng tầng-theo-tầng nên điểm gặp đầu tiên cho tổng ngắn nhất).

**A\*.** Heuristic `h` (tổng khoảng cách vòng) không vượt quá khoảng cách thật và nhất quán (`h(s) ≤ 1 + h(t)` với mọi cạnh `s → t`), nên khi `target` được lấy ra khỏi heap thì `g` là tối ưu.

Cell dưới kiểm chứng cả bốn cách trên các input ngẫu nhiên với oracle độc lập (lặp cập nhật khoảng cách — Cách 1) và cận dưới.

In [21]:
random.seed(752)
def random_case(num_dead):
    dead = {f"{random.randint(0, 9999):04d}" for _ in range(num_dead)}
    target = f"{random.randint(0, 9999):04d}"
    dead.discard(target)
    return sorted(dead), target

count = 0
for _ in range(12):
    dead, target = random_case(random.choice([1, 20, 200, 500]))
    expected = open_lock_relax(dead, target)          # oracle độc lập (Cách 1)
    assert open_lock_bfs(dead, target) == expected
    assert open_lock_bidirectional(dead, target) == expected
    assert open_lock_astar(dead, target) == expected
    assert Solution().openLock(dead, target) == expected
    if expected != -1 and "0000" not in dead:
        assert expected >= lower_bound("0000", target)      # cận dưới
    count += 1
print(count, "input ngẫu nhiên: BFS, hai chiều, A*, Solution khớp oracle")

12 input ngẫu nhiên: BFS, hai chiều, A*, Solution khớp oracle


## 11. Time / Space Complexity

Gọi `N = 10^4` là số trạng thái.

- **Time:** `O(N · 8 · 4)` — mỗi trạng thái được lấy ra khỏi hàng đợi tối đa một lần, sinh 8 hàng xóm, mỗi hàng xóm dựng chuỗi 4 ký tự. Cộng `O(|deadends|)` để dựng `set`. Hằng số lớn nhưng cố định: khoảng `3 · 10^5` phép.
- **Space:** `O(N)` — `visited` và hàng đợi.

So sánh số trạng thái **đã mở rộng** của ba cách trên vài tình huống: gần, xa, và bị bao vây. Ta dùng bản có bộ đếm.

In [22]:
def count_expanded(method, deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1, 0
    expanded = 0
    if method == "bfs":
        visited = dead | {"0000"}
        q = deque([("0000", 0)])
        while q:
            s, d = q.popleft()
            expanded += 1
            for t in neighbors(s):
                if t in visited:
                    continue
                if t == target:
                    return d + 1, expanded
                visited.add(t)
                q.append((t, d + 1))
        return -1, expanded
    if method == "bidir":
        begin, end = {"0000"}, {target}
        visited = set(dead)
        steps = 0
        while begin and end:
            if len(begin) > len(end):
                begin, end = end, begin
            nxt = set()
            for s in begin:
                expanded += 1
                for t in neighbors(s):
                    if t in end:
                        return steps + 1, expanded
                    if t not in visited and t not in begin:
                        nxt.add(t)
            visited |= begin
            begin = nxt
            steps += 1
        return -1, expanded
    if method == "astar":
        def h(s):
            return sum(min(abs(int(a) - int(b)), 10 - abs(int(a) - int(b))) for a, b in zip(s, target))
        best = {"0000": 0}
        heap = [(h("0000"), 0, "0000")]
        done = set()
        while heap:
            f, g, s = heapq.heappop(heap)
            if s in done:
                continue
            done.add(s)
            expanded += 1
            if s == target:
                return g, expanded
            for t in neighbors(s):
                if t in dead:
                    continue
                if g + 1 < best.get(t, INF):
                    best[t] = g + 1
                    heapq.heappush(heap, (g + 1 + h(t), g + 1, t))
        return -1, expanded

cases = {
    "gần (0009)": (["8888"], "0009"),
    "xa (5555)": (["8888"], "5555"),
    "bị bao vây (8888)": examples[2],
}
for name, (dead, target, *_rest) in cases.items():
    row = []
    for method in ("bfs", "bidir", "astar"):
        ans, exp = count_expanded(method, dead, target)
        row.append(f"{method}: ans={ans}, mở rộng {exp}")
    print(f"{name:<20}", " | ".join(row))

gần (0009)           bfs: ans=1, mở rộng 1 | bidir: ans=1, mở rộng 1 | astar: ans=1, mở rộng 2
xa (5555)            bfs: ans=20, mở rộng 9991 | bidir: ans=20, mở rộng 7490 | astar: ans=20, mở rộng 9999
bị bao vây (8888)    bfs: ans=-1, mở rộng 9991 | bidir: ans=-1, mở rộng 2 | astar: ans=-1, mở rộng 9991


Nhận xét: BFS một chiều chi phí tăng theo độ sâu; hai chiều và A\* thường thăm ít hơn đáng kể khi đích xa; còn khi đích bị bao vây, hai chiều dừng ngay vì tập phía đích cạn sớm. Tất cả vẫn nằm trong `O(N)`.

## 12. Bộ test

Gồm: ví dụ đề, edge case, phản ví dụ cho cách sai, đối chiếu các cách (với oracle độc lập) bằng `assert`, và test ngẫu nhiên có seed đúng ràng buộc.

In [23]:
sol = Solution()

# 1) Ví dụ đề
assert sol.openLock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert sol.openLock(["8888"], "0009") == 1
assert sol.openLock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888") == -1
print("ví dụ đề OK")

# 2) Edge case
assert sol.openLock(["0000"], "8888") == -1                 # kẹt ngay từ đầu (đề không cấm!)
assert sol.openLock(["8888"], "0000") == 0                  # đích là điểm xuất phát
assert sol.openLock(["1111"], "0001") == 1
assert sol.openLock(["1111"], "1000") == 1
assert sol.openLock(["1111"], "9000") == 1                  # xoay ngược 0 -> 9
assert sol.openLock(["0001", "0009", "0010", "0090", "0100", "0900", "1000", "9000"], "5555") == -1  # 0000 bị bao vây
assert sol.openLock(["8888"], "5555") == 20                 # xa nhất khi không bị cản
assert sol.openLock(["0001", "1000"], "1001") == 4          # hai đường ngắn (độ dài 2) đều bị chặn, phải đi vòng (độ dài chẵn nên là 4)
assert sol.openLock(["0000"] * 3 + ["1234"], "0001") == -1  # trùng lặp deadends
print("edge case OK")

ví dụ đề OK
edge case OK


In [24]:
# 3) Phản ví dụ cho cách sai (xem mục 13): chỉ xoay một chiều
def wrong_one_direction(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1
    visited = dead | {"0000"}
    q = deque([("0000", 0)])
    if target == "0000":
        return 0
    while q:
        s, d = q.popleft()
        for i in range(4):
            t = s[:i] + str((int(s[i]) + 1) % 10) + s[i + 1:]       # SAI: chỉ +1
            if t in visited:
                continue
            if t == target:
                return d + 1
            visited.add(t)
            q.append((t, d + 1))
    return -1

print("một chiều:", wrong_one_direction(["8888"], "0009"), "(đúng là 1)")
assert wrong_one_direction(["8888"], "0009") == 9 and sol.openLock(["8888"], "0009") == 1

# 4) Đối chiếu 4 cách + oracle độc lập trên các deadends ngẫu nhiên
random.seed(2024)
impls = [open_lock_bfs, open_lock_bidirectional, open_lock_astar]
classes = [SolutionBFS(), SolutionBidirectional(), SolutionAStar(), Solution()]
for trial in range(40):
    dead, target = random_case(random.choice([1, 5, 50, 300, 500]))
    if random.random() < 0.1:
        dead = sorted(set(dead) | {"0000"} - {target})              # thỉnh thoảng 0000 là ngõ cụt
    base = open_lock_bfs(dead, target)
    for f in impls:
        assert f(dead, target) == base, (f.__name__, dead[:5], target)
    for c in classes:
        assert c.openLock(dead, target) == base
print("40 input ngẫu nhiên: các cách khớp nhau")

một chiều: 9 (đúng là 1)


40 input ngẫu nhiên: các cách khớp nhau


In [25]:
# 5) Oracle độc lập (lặp cập nhật khoảng cách) trên một số input ngẫu nhiên; ngõ cụt dày để có đường vòng/chặn
random.seed(99)
for trial in range(10):
    dead, target = random_case(random.choice([100, 500]))
    expected = open_lock_relax(dead, target)
    assert sol.openLock(dead, target) == expected
print("10 input: Solution khớp oracle lặp cập nhật")

# 6) Test bao vây: bao quanh target bằng 8 ngõ cụt -> -1; bao quanh 0000 -> -1
random.seed(5)
for _ in range(50):
    t = f"{random.randint(1, 9999):04d}"
    ring = neighbors(t)
    assert sol.openLock(ring, t) == -1        # bị bao vây (hoặc 0000 nằm trong ring): luôn -1
print("bao vây target: luôn -1")

# 7) Test tính chất: không có ngõ cụt chặn đường thì đáp án = cận dưới; luôn >= cận dưới
random.seed(7)
for _ in range(200):
    target = f"{random.randint(0, 9999):04d}"
    far = f"{(int(target) + 4444) % 10000:04d}"              # một ngõ cụt tuỳ ý, thường không nằm trên đường ngắn nhất
    if far == target or far == "0000":
        continue
    ans = sol.openLock([far], target)
    assert ans >= lower_bound("0000", target)
assert sol.openLock(["9999"], "0202") == lower_bound("0000", "0202") == 4
print("test cận dưới OK")

10 input: Solution khớp oracle lặp cập nhật


bao vây target: luôn -1


test cận dưới OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp `"0000"` nằm trong `deadends`

Đề chỉ đảm bảo `target` không phải ngõ cụt; `"0000"` thì **có thể**. Khi đó khoá kẹt ngay và đáp án phải là `-1`. Nhiều lời giải chỉ đưa `"0000"` vào hàng đợi mà không kiểm tra, trả kết quả sai.

In [26]:
def wrong_no_check_start(deadends, target):
    dead = set(deadends)
    visited = {"0000"}                         # SAI: bỏ quên "0000" có thể là ngõ cụt; cũng chưa nạp dead vào visited ngay
    q = deque([("0000", 0)])
    if target == "0000":
        return 0
    while q:
        s, d = q.popleft()
        for t in neighbors(s):
            if t in visited or t in dead:
                continue
            if t == target:
                return d + 1
            visited.add(t)
            q.append((t, d + 1))
    return -1

print("sai :", wrong_no_check_start(["0000"], "0001"), "(kỳ vọng -1)")
print("đúng:", Solution().openLock(["0000"], "0001"))

sai : 1 (kỳ vọng -1)
đúng: -1


**Vì sao sai:** khoá đã kẹt ở `"0000"`, không xoay được. Code sai vẫn xoay bình thường. **Quy tắc:** kiểm tra `"0000" in dead` đầu tiên.

### Lỗi 2 — Đánh dấu `visited` khi lấy ra khỏi hàng đợi (quá muộn)

Kết quả vẫn đúng nhưng hàng đợi phình to vì cùng một trạng thái được thêm nhiều lần; có thể TLE với các cách làm nặng hơn. Cell dưới đo số lần phần tử được đưa vào hàng đợi.

In [27]:
def pushes_mark_late(deadends, target):
    dead = set(deadends)
    visited = set(dead)
    q = deque([("0000", 0)])
    pushes = 1
    while q:
        s, d = q.popleft()
        if s in visited:
            continue
        visited.add(s)                          # đánh dấu khi LẤY RA: quá muộn
        if s == target:
            return d, pushes
        for t in neighbors(s):
            if t not in visited:
                q.append((t, d + 1))
                pushes += 1
    return -1, pushes

def pushes_mark_early(deadends, target):
    dead = set(deadends)
    visited = dead | {"0000"}
    q = deque([("0000", 0)])
    pushes = 1
    while q:
        s, d = q.popleft()
        if s == target:
            return d, pushes
        for t in neighbors(s):
            if t not in visited:
                visited.add(t)                  # đánh dấu khi ĐƯA VÀO
                q.append((t, d + 1))
                pushes += 1
    return -1, pushes

dead, target = ["8888"], "5555"
print("đánh dấu muộn :", pushes_mark_late(dead, target))
print("đánh dấu sớm  :", pushes_mark_early(dead, target))

đánh dấu muộn : (20, 39993)
đánh dấu sớm  : (20, 9999)


**Vì sao sai (chậm):** một trạng thái có tới 8 hàng xóm, nên bị đưa vào hàng đợi tới 8 lần. **Quy tắc:** *"thêm vào hàng đợi thì đánh dấu luôn"*.

### Lỗi 3 — Chỉ xoay một chiều, quên quay vòng

Chỉ xoay `+1` thì `"0000" → "0009"` mất 9 nấc thay vì 1 (mục 12 đã đo). Phải xoay cả `+1` và `-1`, và dùng `% 10` để `0 ↔ 9`.

### Lỗi 4 — Dùng DFS thay BFS

DFS (đi sâu) tìm ra **một** đường nhưng không đảm bảo ngắn nhất.

In [28]:
def wrong_dfs(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1
    visited = set(dead) | {"0000"}
    stack = [("0000", 0)]                      # DFS: lấy phần tử CUỐI (ngăn xếp) thay vì đầu (hàng đợi)
    while stack:
        s, d = stack.pop()
        for t in neighbors(s):
            if t in visited:
                continue
            if t == target:
                return d + 1                   # đường đầu tiên tìm thấy, chưa chắc ngắn nhất
            visited.add(t)
            stack.append((t, d + 1))
    return -1

print("DFS :", wrong_dfs(["8888"], "0202"), "(đúng là 4)")
print("BFS :", Solution().openLock(["8888"], "0202"))

DFS : 266 (đúng là 4)
BFS : 4


**Vì sao sai:** DFS đi sâu theo hàng xóm đầu tiên rồi mới quay lại, trả về đường **đầu tiên tìm thấy**, thường rất dài. BFS duyệt theo tầng nên đường đầu tiên chính là ngắn nhất.

### Lỗi 5 — Dùng `str.replace` để đổi chữ số

`s.replace(old, new)` đổi **mọi** ký tự bằng `old`, không chỉ ký tự ở vị trí `i`. Với `"0000"`, `replace("0", "1")` cho `"1111"` — không phải hàng xóm.

In [29]:
s = "0000"
print("replace:", s.replace("0", "1"))                 # 1111 -> SAI
print("ghép   :", s[:2] + "1" + s[3:])                # 0010 -> đúng (đổi bánh thứ 3)

def wrong_replace_neighbors(s):
    res = []
    for i in range(4):
        d = int(s[i])
        for delta in (1, -1):
            res.append(s.replace(s[i], str((d + delta) % 10)))   # SAI
    return res

print(wrong_replace_neighbors("0000")[:4])             # toàn '1111' / '9999'

replace: 1111
ghép   : 0010
['1111', '9999', '1111', '9999']


**Cách đúng:** `s[:i] + str((d + delta) % 10) + s[i + 1:]`.

## 14. Mẹo & nhận dạng pattern

- **"Số bước ít nhất để biến trạng thái này thành trạng thái kia, mỗi bước chi phí bằng nhau"** → BFS trên đồ thị trạng thái. Việc đầu tiên là xác định **đỉnh** (trạng thái), **cạnh** (phép biến đổi), **đỉnh bị cấm** (ngõ cụt).
- **Ngõ cụt/vật cản** → cho vào `visited` từ đầu (đỡ phải kiểm tra riêng).
- **Đánh dấu khi đưa vào hàng đợi**, không phải khi lấy ra.
- **Trạng thái ít (`10^4`), hàng xóm ít (8)** → BFS thẳng đủ nhanh; nếu không gian lớn hơn, nghĩ tới BFS hai chiều hoặc A\*.
- **Hai chiều** hiệu quả khi biết rõ cả điểm bắt đầu và điểm kết thúc và đồ thị vô hướng; mở rộng phía có **tập nhỏ hơn**.
- Bài họ hàng: Word Ladder (127), Minimum Genetic Mutation (433), Sliding Puzzle (773), Minimum Knight Moves (1197), Jump Game IV (1345).

## 15. Biến thể

### 15.1. In ra đường đi (không chỉ độ dài)

Lưu `parent[t] = s` khi đánh dấu `t` lần đầu rồi truy ngược từ `target` về `"0000"`.

In [30]:
def open_lock_path(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return None
    if target == "0000":
        return ["0000"]
    parent = {"0000": None}
    visited = dead | {"0000"}
    q = deque(["0000"])
    while q:
        s = q.popleft()
        for t in neighbors(s):
            if t in visited:
                continue
            parent[t] = s
            if t == target:
                path = [t]
                while parent[path[-1]] is not None:
                    path.append(parent[path[-1]])
                return path[::-1]
            visited.add(t)
            q.append(t)
    return None

p = open_lock_path(examples[0][0], examples[0][1])
print(p, "-> số lần xoay =", len(p) - 1)
dead_set = set(examples[0][0])
assert all(x not in dead_set for x in p)                  # không đi qua ngõ cụt
assert all(sum(a != b for a, b in zip(u, v)) == 1 for u, v in zip(p, p[1:]))   # mỗi bước đổi đúng một bánh
assert len(p) - 1 == Solution().openLock(*examples[0][:2])
print(open_lock_path(["8888"], "0009"), open_lock_path(["0000"], "0009"))

['0000', '1000', '1100', '1200', '1201', '1202', '0202'] -> số lần xoay = 6
['0000', '0009'] None


### 15.2. Tổng quát hoá: `W` bánh xe, mỗi bánh `K` chữ số

Chỉ cần tham số hoá độ dài chuỗi và modulo `K`. Với `W = 2, K = 10` ta có khoá 2 bánh (100 trạng thái) — vừa đủ nhỏ để kiểm chứng bằng oracle vét cạn. Hàm dưới dùng BFS chuẩn, còn oracle là Floyd-Warshall trên đồ thị 100 đỉnh.

In [31]:
def open_lock_general(deadends, target, W=4, K=10):
    dead = set(deadends)
    start = "0" * W
    if start in dead:
        return -1
    if target == start:
        return 0
    visited = dead | {start}
    q = deque([(start, 0)])
    while q:
        s, d = q.popleft()
        for i in range(W):
            c = int(s[i])
            for delta in (1, -1):
                t = s[:i] + str((c + delta) % K) + s[i + 1:]
                if t in visited:
                    continue
                if t == target:
                    return d + 1
                visited.add(t)
                q.append((t, d + 1))
    return -1

def floyd_oracle(deadends, target, W, K):
    # Floyd-Warshall trên đồ thị đủ K**W đỉnh, bỏ đỉnh ngõ cụt
    nodes = all_codes(W, K)
    idx = {s: i for i, s in enumerate(nodes)}
    dead = set(deadends)
    n = len(nodes)
    INF_ = 10 ** 9
    D = [[INF_] * n for _ in range(n)]
    for s in nodes:
        if s in dead:
            continue
        D[idx[s]][idx[s]] = 0
        for i in range(W):
            c = int(s[i])
            for delta in (1, -1):
                t = s[:i] + str((c + delta) % K) + s[i + 1:]
                if t not in dead:
                    D[idx[s]][idx[t]] = 1
    for k in range(n):
        Dk = D[k]
        for i in range(n):
            dik = D[i][k]
            if dik == INF_:
                continue
            Di = D[i]
            for j in range(n):
                if dik + Dk[j] < Di[j]:
                    Di[j] = dik + Dk[j]
    start = "0" * W
    if start in dead:
        return -1
    r = D[idx[start]][idx[target]]
    return -1 if r >= INF_ else r

from itertools import product

def all_codes(W, K):
    return ["".join(map(str, c)) for c in product(range(K), repeat=W)]

random.seed(15)
for _ in range(8):
    W, K = 2, random.choice([3, 5, 10])
    nodes = all_codes(W, K)
    target = random.choice(nodes)
    dead = random.sample([x for x in nodes if x != target], random.randint(0, len(nodes) // 3))
    assert open_lock_general(dead, target, W, K) == floyd_oracle(dead, target, W, K)
print("tổng quát hoá khớp Floyd-Warshall (W = 2, K tuỳ ý)")
print("3 bánh, K = 10, target 555:", open_lock_general(["999"], "555", 3, 10))

tổng quát hoá khớp Floyd-Warshall (W = 2, K tuỳ ý)
3 bánh, K = 10, target 555: 15


### 15.3. Đếm số đường ngắn nhất

Trong BFS, ngoài `dist[v]` ta giữ `ways[v]` = số đường ngắn nhất tới `v`. Khi thấy hàng xóm `t` ở **tầng kế tiếp** (`dist[t] == dist[s] + 1`), cộng `ways[t] += ways[s]`. Đây là Bài 5 ở mục 17.

### 15.4. Bánh chỉ xoay một chiều

Nếu mỗi bánh chỉ xoay **lên** (`+1`), đồ thị trở thành có hướng; BFS vẫn dùng được, chỉ bỏ nửa số hàng xóm. Đáp án không còn đối xứng (`"0009"` mất 9 nấc).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lặp cập nhật khoảng cách | Bellman-Ford trên 10^4 trạng thái | `O(L · N · 8)` | `O(N)` | Oracle độc lập để kiểm tra |
| 2. BFS `deque` | Duyệt theo tầng, `visited` | `O(N · 8 · 4)` | `O(N)` | **Lời giải chuẩn nộp LeetCode** |
| 3. BFS hai chiều | Hai biên gặp nhau, luôn mở rộng phía nhỏ | `O(N · 8 · 4)` xấu nhất, thường ít hơn nhiều | `O(N)` | Đồ thị lớn, biết cả hai đầu; đích bị bao vây |
| 4. A\* | `f = g + h`, `h` = tổng khoảng cách vòng | `O(N · 8 · log N)` | `O(N)` | Có heuristic tốt; học kỹ thuật tìm kiếm có thông tin |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(len(neighbors("5555")))
print(sorted(neighbors("0900"))[:3])
print(Solution().openLock(["1111"], "9000"))
```

In [32]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Khi **không có ngõ cụt**, mỗi tầng BFS chứa bao nhiêu trạng thái? Hãy tính tay: tầng 1 có bao nhiêu (đổi một bánh một nấc, 2 chiều)? Tầng 2 có bao nhiêu? (Gợi ý: hai bánh khác nhau mỗi bánh ±1, hoặc cùng một bánh đi ±2 — chú ý `+1` rồi `-1` quay về tầng 0.) Và số nấc tối thiểu từ `"0000"` tới `"3728"` không ngõ cụt là bao nhiêu?

In [33]:
# Bài 2: điền kết quả tính tay
hand_level1 = None
hand_level2 = None
hand_dist_3728 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS.

In [34]:
def exercise_3_open_lock(deadends: List[str], target: str) -> int:
    # TODO 1: dead = set(deadends); nếu "0000" trong dead -> -1; nếu target == "0000" -> 0
    # TODO 2: visited = dead | {"0000"}; queue = deque([("0000", 0)])
    # TODO 3: lấy (s, d) ở đầu; với mỗi hàng xóm t (dùng neighbors(s)) chưa visited:
    #     TODO 4: nếu t == target -> return d + 1
    #     TODO 5: visited.add(t) và đưa (t, d + 1) vào queue
    # TODO 6: hết queue -> return -1
    return None


# assert exercise_3_open_lock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
# assert exercise_3_open_lock(["8888"], "0009") == 1
# assert exercise_3_open_lock(["0000"], "0001") == -1

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_bidirectional(deadends, target)`: BFS hai chiều, luôn mở rộng tập nhỏ hơn.

In [35]:
def exercise_4_bidirectional(deadends: List[str], target: str) -> int:
    # TODO
    pass


# assert exercise_4_bidirectional(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
# assert exercise_4_bidirectional(["8888"], "0009") == 1
# assert exercise_4_bidirectional(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888") == -1

### Bài 5 — Mini challenge: đếm số đường ngắn nhất (khó)

Viết `exercise_5_count_paths(deadends, target, W=4, K=10)` trả về **số đường đi có độ dài ngắn nhất** từ `"0" * W` tới `target` (một đường là dãy các trạng thái); `0` nếu không tới được. Quy ước: nếu `target == "0" * W` thì có `1` đường (độ dài 0); nếu `"0" * W` là ngõ cụt thì `0`.

Ví dụ: `W = 2, K = 10, deadends = [], target = "11"` → `2` (đi `01` rồi `11` hoặc `10` rồi `11`). `target = "02"` → `1`.

In [36]:
def exercise_5_count_paths(deadends: List[str], target: str, W: int = 4, K: int = 10) -> int:
    # TODO
    pass


# assert exercise_5_count_paths([], "11", W=2) == 2
# assert exercise_5_count_paths([], "02", W=2) == 1
# assert exercise_5_count_paths(["01"], "11", W=2) == 1

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Coi mỗi mã là một đỉnh; hàng xóm là 8 mã sau một lần xoay. Ngõ cụt là đỉnh cấm.
- *Mức 2 (vừa):* BFS: hàng đợi `deque`; đưa ngõ cụt vào `visited` ngay từ đầu; đánh dấu `visited` khi **đưa vào** hàng đợi. Hai chiều: giữ hai tập `begin`, `end`; luôn mở rộng tập nhỏ hơn; hàng xóm thuộc tập bên kia thì gặp nhau.
- *Mức 3 (gần đáp án):* BFS: `if t == target: return d + 1`; hai chiều: `if t in end: return steps + 1`, `visited |= begin; begin = nxt; steps += 1`.

**Bài 5**

- *Mức 1:* BFS theo tầng vẫn dùng được; thay vì chỉ đánh dấu "đã thăm", lưu khoảng cách `dist`.
- *Mức 2:* Lưu `ways[v]`. Khi thấy hàng xóm `t`: nếu chưa có `dist[t]` thì `dist[t] = dist[s] + 1`, `ways[t] = ways[s]`; nếu `dist[t] == dist[s] + 1` thì `ways[t] += ways[s]`.
- *Mức 3:* Duyệt đến hết (hoặc đến khi tầng chứa `target` xử lý xong) rồi trả `ways[target]` (`0` nếu `target` không có trong `dist`).

## 19. Đáp án tham khảo

### Đáp án Bài 1

- `len(neighbors("5555")) = 8`.
- `neighbors("0900")` gồm `1900, 9900` (bánh 1), `0000, 0800` (bánh 2 là `9`: lên quay vòng về `0`, xuống thành `8`), `0910, 0990` (bánh 3), `0901, 0909` (bánh 4). Sắp xếp theo thứ tự từ điển, ba chuỗi đầu là `0000, 0800, 0901`.
- `openLock(["1111"], "9000") = 1` (xoay ngược bánh đầu).

In [37]:
print(sorted(neighbors("0900")))
assert len(neighbors("5555")) == 8
assert sorted(neighbors("0900"))[:3] == ["0000", "0800", "0901"]
assert Solution().openLock(["1111"], "9000") == 1
print("Bài 1 OK")

['0000', '0800', '0901', '0909', '0910', '0990', '1900', '9900']
Bài 1 OK


### Đáp án Bài 2

Tầng 1: mỗi bánh 2 chiều → `4 × 2 = 8`. Tầng 2: hai bánh khác nhau, mỗi bánh ±1: `C(4, 2) × 2 × 2 = 24`; cùng một bánh đi hai nấc cùng chiều (`+2` hoặc `-2`): `4 × 2 = 8`; tổng `32`. Số nấc tới `"3728"`: bánh `0→3 = 3`, `0→7 = min(7, 3) = 3`, `0→2 = 2`, `0→8 = min(8, 2) = 2` → `10`.

In [38]:
hand_level1, hand_level2, hand_dist_3728 = 8, 32, 10
sz = level_sizes()
assert sz[1] == hand_level1 and sz[2] == hand_level2
assert lower_bound("0000", "3728") == hand_dist_3728 == Solution().openLock(["9999"], "3728")
print("Bài 2 OK", sz[:4])

Bài 2 OK [1, 8, 32, 88]


### Đáp án Bài 3

In [39]:
def exercise_3_solution(deadends: List[str], target: str) -> int:
    dead = set(deadends)
    if "0000" in dead:
        return -1
    if target == "0000":
        return 0
    visited = dead | {"0000"}
    queue = deque([("0000", 0)])
    while queue:
        s, d = queue.popleft()
        for t in neighbors(s):
            if t in visited:
                continue
            if t == target:
                return d + 1
            visited.add(t)
            queue.append((t, d + 1))
    return -1

assert exercise_3_solution(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert exercise_3_solution(["8888"], "0009") == 1
assert exercise_3_solution(["0000"], "0001") == -1
random.seed(3)
for _ in range(25):
    dead, target = random_case(random.choice([1, 50, 300]))
    assert exercise_3_solution(dead, target) == Solution().openLock(dead, target)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_solution(deadends: List[str], target: str) -> int:
    dead = set(deadends)
    if "0000" in dead:
        return -1
    if target == "0000":
        return 0
    begin, end = {"0000"}, {target}
    visited = set(dead)
    steps = 0
    while begin and end:
        if len(begin) > len(end):
            begin, end = end, begin
        nxt = set()
        for s in begin:
            for t in neighbors(s):
                if t in end:
                    return steps + 1
                if t not in visited and t not in begin:
                    nxt.add(t)
        visited |= begin
        begin = nxt
        steps += 1
    return -1

assert exercise_4_solution(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert exercise_4_solution(["8888"], "0009") == 1
assert exercise_4_solution(examples[2][0], "8888") == -1
random.seed(4)
for _ in range(25):
    dead, target = random_case(random.choice([1, 50, 300]))
    assert exercise_4_solution(dead, target) == Solution().openLock(dead, target)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_solution(deadends: List[str], target: str, W: int = 4, K: int = 10) -> int:
    dead = set(deadends)
    start = "0" * W
    if start in dead:
        return 0
    dist = {start: 0}
    ways = {start: 1}
    q = deque([start])
    while q:
        s = q.popleft()
        for i in range(W):
            c = int(s[i])
            for delta in (1, -1):
                t = s[:i] + str((c + delta) % K) + s[i + 1:]
                if t in dead:
                    continue
                if t not in dist:
                    dist[t] = dist[s] + 1
                    ways[t] = ways[s]
                    q.append(t)
                elif dist[t] == dist[s] + 1:
                    ways[t] += ways[s]
    return ways.get(target, 0)

assert exercise_5_solution([], "11", W=2) == 2
assert exercise_5_solution([], "02", W=2) == 1
assert exercise_5_solution(["01"], "11", W=2) == 1
assert exercise_5_solution([], "00", W=2) == 1

# Kiểm chứng bằng liệt kê trực tiếp mọi đường đơn có độ dài đúng bằng khoảng cách ngắn nhất (DFS) trên khoá 2 bánh
def count_by_dfs(deadends, target, W, K):
    dead = set(deadends)
    start = "0" * W
    if start in dead:
        return 0
    best = open_lock_general(deadends, target, W, K)
    if best == -1:
        return 0
    def dfs(s, left, seen):
        if left == 0:
            return 1 if s == target else 0
        total = 0
        for i in range(W):
            c = int(s[i])
            for delta in (1, -1):
                t = s[:i] + str((c + delta) % K) + s[i + 1:]
                if t in dead or t in seen:
                    continue
                total += dfs(t, left - 1, seen | {t})
        return total
    return dfs(start, best, {start})

random.seed(55)
for _ in range(60):
    W, K = 2, random.choice([3, 4, 5, 10])
    nodes = all_codes(W, K)
    target = random.choice(nodes)
    dead = random.sample([x for x in nodes if x != target], random.randint(0, len(nodes) // 3))
    assert exercise_5_solution(dead, target, W, K) == count_by_dfs(dead, target, W, K), (dead, target, K)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def openLock(self, deadends: List[str], target: str) -> int:
        dead = set(deadends)
        if "0000" in dead:
            return -1
        if target == "0000":
            return 0
        visited = dead | {"0000"}
        queue = deque([("0000", 0)])
        while queue:
            s, d = queue.popleft()
            for i in range(4):
                c = int(s[i])
                for delta in (1, -1):
                    t = s[:i] + str((c + delta) % 10) + s[i + 1:]
                    if t in visited:
                        continue
                    if t == target:
                        return d + 1
                    visited.add(t)
                    queue.append((t, d + 1))
        return -1
```

Câu thần chú: **"Đỉnh = trạng thái, cạnh = một lần xoay, ngõ cụt = đã thăm; BFS theo tầng, thêm vào hàng đợi là đánh dấu luôn."**

### Checklist trước khi submit

- [ ] Kiểm tra `"0000" in deadends` → `-1`; và `target == "0000"` → `0`.
- [ ] Ngõ cụt vào `visited` ngay từ đầu.
- [ ] Hàng xóm: 4 bánh × 2 chiều, quay vòng bằng `% 10`; dùng cắt/ghép chuỗi, không dùng `replace`.
- [ ] Đánh dấu `visited` khi **đưa vào** hàng đợi.
- [ ] Dùng `deque.popleft()` (không dùng `list.pop(0)`).
- [ ] Dùng BFS (không phải DFS) để có đường ngắn nhất.
- [ ] Đã thử: đích sát (`1` bước), xoay vòng `0 ↔ 9`, đích bị bao vây (`-1`), `"0000"` là ngõ cụt.
- [ ] Đã đối chiếu oracle độc lập (lặp cập nhật khoảng cách) trên input ngẫu nhiên.
- [ ] Skeleton đúng: `class Solution`, method `openLock(self, deadends: List[str], target: str) -> int`.